In [5]:
# ===== Célula 1 — Carregar perguntas, rótulos do juiz e textos =====
import random
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
from src.avaliacao.rotular_relevancia import carregar_rotulos
from src.avaliacao.validar_perguntas import carregar_perguntas

perguntas = {p["id"]: p for p in carregar_perguntas()}
rotulos = carregar_rotulos()  # {(id_pergunta, chave_texto): nota}
corpus = pd.read_parquet(RAIZ / "data" / "processed" / "corpus.parquet")
texto_por_chave = corpus.drop_duplicates("chave_texto").set_index("chave_texto")["texto"]

print(f"{len(perguntas)} perguntas | {len(rotulos):,} rótulos do juiz")
print("Distribuição:", dict(sorted(Counter(rotulos.values()).items())))

31 perguntas | 2,222 rótulos do juiz
Distribuição: {0: 586, 1: 422, 2: 1214}


In [2]:
# ===== Célula 2 — Sortear a amostra para validação humana (sem mostrar a nota do juiz) =====
random.seed(42)  # sorteio reprodutível: qualquer pessoa obtém a mesma amostra

# Os 2 casos suspeitos: notas 2 em perguntas que deveriam ser recusadas
suspeitos = [par for par, nota in rotulos.items()
             if nota == 2 and perguntas[par[0]]["esperado"] == "recusar"]

amostra = list(suspeitos)
for nota in (2, 1, 0):
    candidatos = [par for par, n in rotulos.items() if n == nota and par not in amostra]
    faltam = 7 - sum(1 for par in amostra if rotulos[par] == nota)
    amostra += random.sample(candidatos, faltam)
random.shuffle(amostra)  # embaralha para as notas do juiz não ficarem em sequência

for i, (id_pergunta, chave) in enumerate(amostra, start=1):
    p = perguntas[id_pergunta]
    filtros = f" | filtros: {p['filtros']}" if p.get("filtros") else ""
    print(f"\n#{i:02d} [{id_pergunta}] PERGUNTA: {p['pergunta']}{filtros}")
    print(f"     AVALIAÇÃO: {texto_por_chave[chave]}")


#01 [P28] PERGUNTA: O que os clientes dizem sobre pagar as compras com criptomoedas?
     AVALIAÇÃO: Excelente 👍🏽. Respeito ao cliente, qualidade, satisfação. Cliente Feliz!!!

#02 [P29] PERGUNTA: O que os clientes relatam sobre a entrega feita por drones?
     AVALIAÇÃO: Entrega no prazo e o produto exatamente o que havia visualizado e comprado. Tudo cem por cento. Todas as caracteristicas divulgada do produto corresponderam. Muito bem embalado e fiquei 100% satisfeit

#03 [P02] PERGUNTA: O que os clientes relatam sobre problemas relacionados à entrega?
     AVALIAÇÃO: Muita demora pra entregar e acima de tudo nao tem nehuma informacao se o produto saiu pra entrega ou ate mesmo qual é a situação q o produto está!! E Vale lembrar que o pagamento já foi efetuado!

#04 [P29] PERGUNTA: O que os clientes relatam sobre a entrega feita por drones?
     AVALIAÇÃO: Entrega super rápida. Muito eficente a entrega de vcs,fiquei impressionada com a agilidade,parabéns a vcs.Mega recomendo!

#05 [P

In [3]:
# ===== Célula 3 — Suas notas e a comparação com o juiz =====
# Preencha na ordem #01, #02, ..., #21 (2 = responde diretamente, 1 = mesmo assunto, 0 = sem relação)
notas_humanas = [0, 2, 2, 2, 1, 0, 2,
                 2, 2, 0, 0, 2, 1, 1,
                 2, 2, 1, 2, 1, 1, 0]

assert len(notas_humanas) == len(amostra), "a lista precisa ter uma nota para cada par"
assert all(n in (0, 1, 2) for n in notas_humanas), "use apenas 0, 1 ou 2"

notas_juiz = [rotulos[par] for par in amostra]
validacao = pd.DataFrame({
    "n": range(1, len(amostra) + 1),
    "id_pergunta": [par[0] for par in amostra],
    "texto": [texto_por_chave[par[1]] for par in amostra],
    "nota_humana": notas_humanas,
    "nota_juiz": notas_juiz,
})

# Concordância exata e "relevante (2) ou não"
exata = (validacao["nota_humana"] == validacao["nota_juiz"]).mean()
binaria = ((validacao["nota_humana"] == 2) == (validacao["nota_juiz"] == 2)).mean()

# Kappa de Cohen: concordância descontando a que aconteceria por acaso
p_humano = validacao["nota_humana"].value_counts(normalize=True)
p_juiz = validacao["nota_juiz"].value_counts(normalize=True)
acaso = sum(p_humano.get(k, 0) * p_juiz.get(k, 0) for k in (0, 1, 2))
kappa = (exata - acaso) / (1 - acaso)

print(f"Concordância exata:              {exata:.0%}")
print(f"Concordância 'relevante ou não': {binaria:.0%}")
print(f"Kappa de Cohen:                  {kappa:.2f}")
print("\nMatriz de confusão (linhas = você, colunas = juiz):")
print(pd.crosstab(validacao["nota_humana"], validacao["nota_juiz"],
                  rownames=["você"], colnames=["juiz"]))

print("\nPares em que vocês discordaram:")
discordancias = validacao[validacao["nota_humana"] != validacao["nota_juiz"]]
for _, d in discordancias.iterrows():
    print(f"  #{d['n']:02d} [{d['id_pergunta']}] você {d['nota_humana']} x juiz {d['nota_juiz']} | {d['texto'][:90]}")

validacao.to_csv(RAIZ / "resultados" / "validacao_relevancia.csv", index=False, encoding="utf-8")
print("\nValidação salva em resultados/validacao_relevancia.csv")

Concordância exata:              52%
Concordância 'relevante ou não': 57%
Kappa de Cohen:                  0.29

Matriz de confusão (linhas = você, colunas = juiz):
juiz  0  1  2
você         
0     4  0  1
1     1  3  2
2     2  4  4

Pares em que vocês discordaram:
  #02 [P29] você 2 x juiz 1 | Entrega no prazo e o produto exatamente o que havia visualizado e comprado. Tudo cem por c
  #04 [P29] você 2 x juiz 1 | Entrega super rápida. Muito eficente a entrega de vcs,fiquei impressionada com a agilidade
  #08 [P18] você 2 x juiz 1 | Recomendo a todos os clientes
  #11 [P04] você 0 x juiz 2 | Sempre gostei de comprar nas lannister, mas essa entrega deixou a desejar.
  #12 [P30] você 2 x juiz 0 | Recomendo para todos que eu conheço !! Ótima loja virtual !! Vocês estão de parabéns!!
  #13 [P11] você 1 x juiz 2 | produto com itens faltando
  #15 [P05] você 2 x juiz 0 | Problems existem, porem, necessario compreende-los.
  #16 [P05] você 2 x juiz 1 | Prazo. Há diferença de prazo de entrega

In [4]:
# ===== Célula 4 — Notas de consenso (após revisar as discordâncias) e nova comparação =====
# Partindo das notas humanas; mudar apenas os pares revisados.
notas_consenso = list(notas_humanas)
ajustes = {  # número do par: nota de consenso
    2: 1, 4: 1, 8: 1, 11: 2, 12: 0, 13: 2, 15: 0,  # pela régua, nota do juiz
    16: 2, 20: 1,                                   # pela régua, nota humana
    19: 1,                                          # ambíguo: decisão do revisor
}
for n, nota in ajustes.items():
    notas_consenso[n - 1] = nota


def concordancia(a: list[int], b: list[int]) -> dict:
    """Concordância exata, binária (2 ou não) e kappa de Cohen entre duas listas de notas."""
    a, b = pd.Series(a), pd.Series(b)
    exata = (a == b).mean()
    binaria = ((a == 2) == (b == 2)).mean()
    pa, pb = a.value_counts(normalize=True), b.value_counts(normalize=True)
    acaso = sum(pa.get(k, 0) * pb.get(k, 0) for k in (0, 1, 2))
    return {"exata": exata, "binaria": binaria, "kappa": (exata - acaso) / (1 - acaso)}


antes = concordancia(notas_humanas, notas_juiz)
depois = concordancia(notas_consenso, notas_juiz)
print(f"{'':28s}{'1ª rodada (às cegas)':>22s}{'após consenso':>16s}")
print(f"{'Concordância exata':28s}{antes['exata']:>22.0%}{depois['exata']:>16.0%}")
print(f"{'Concordância relevante/não':28s}{antes['binaria']:>22.0%}{depois['binaria']:>16.0%}")
print(f"{'Kappa de Cohen':28s}{antes['kappa']:>22.2f}{depois['kappa']:>16.2f}")

print("\nMatriz de confusão após consenso (linhas = consenso, colunas = juiz):")
print(pd.crosstab(pd.Series(notas_consenso, name="consenso"), pd.Series(notas_juiz, name="juiz")))

validacao["nota_consenso"] = notas_consenso
validacao.to_csv(RAIZ / "resultados" / "validacao_relevancia.csv", index=False, encoding="utf-8")
print("\nValidação (com consenso) salva em resultados/validacao_relevancia.csv")

                              1ª rodada (às cegas)   após consenso
Concordância exata                             52%             86%
Concordância relevante/não                     57%             90%
Kappa de Cohen                                0.29            0.79

Matriz de confusão após consenso (linhas = consenso, colunas = juiz):
juiz      0  1  2
consenso         
0         6  0  0
1         1  6  1
2         0  1  6

Validação (com consenso) salva em resultados/validacao_relevancia.csv


In [6]:
# ===== Célula 5 — Analisar as respostas geradas (sem gastar API) =====
import json

from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)
arquivo = RAIZ / "resultados" / "respostas" / "densa_k20_prompt_v2.jsonl"
registros = [json.loads(l) for l in arquivo.read_text(encoding="utf-8").splitlines() if l.strip()]
respostas_df = pd.DataFrame([r for r in registros if "erro" not in r])

# 1) Trechos sinalizados pelo verificador, com o tamanho (em palavras) de cada um
linhas = [{"modelo": r["modelo"].split("/")[-1], "id": r["id"],
           "palavras": len(t.split()), "trecho": t}
          for _, r in respostas_df.iterrows() for t in r["trechos_nao_encontrados"]]
trechos = pd.DataFrame(linhas)
print("Tamanho dos trechos sinalizados, por modelo:")
print(trechos.groupby("modelo")["palavras"].describe()[["count", "min", "50%", "max"]])

print("\nTrechos de 1 palavra (candidatos a 'aspas de destaque'):")
print(trechos[trechos["palavras"] == 1][["modelo", "id", "trecho"]].to_string(index=False))

print("\nTodos os trechos sinalizados do Qwen:")
print(trechos[trechos["modelo"].str.startswith("qwen")][["id", "trecho"]].to_string(index=False))

print("\nAmostra de 10 trechos sinalizados do gpt-oss:")
gpt = trechos[trechos["modelo"].str.startswith("gpt")]
print(gpt.sample(min(10, len(gpt)), random_state=42)[["id", "trecho"]].to_string(index=False))

# 2) A resposta do gpt-oss que não recusou a pergunta sobre o roteador
p26 = respostas_df[(respostas_df["id"] == "P26")
                   & respostas_df["modelo"].str.contains("gpt-oss")].iloc[0]
display(Markdown(f"---\n### P26 — gpt-oss-120b\n{p26['resposta']}"))

Tamanho dos trechos sinalizados, por modelo:
              count  min  50%   max
modelo                             
gpt-oss-120b   31.0  2.0  5.0  12.0
qwen3.8-27b     5.0  1.0  2.0   5.0

Trechos de 1 palavra (candidatos a 'aspas de destaque'):
     modelo  id    trecho
qwen3.8-27b P03 eficiente

Todos os trechos sinalizados do Qwen:
 id                   trecho
P03                eficiente
P07 produtos [são] originais
P20     47 dias após o prazo
P30         provador virtual
P30      realidade aumentada

Amostra de 10 trechos sinalizados do gpt-oss:
 id                                                               trecho
P19                Atendimento péssimo… não tem ninguém… para reclamação
P07                                          chegaram bem antes do prazo
P15 o produto sofreu extravio no caminho … a loja prontamente resolveram
P08                                                 60 dias após o prazo
P06                                           entrega atrasado (3 dias…)
P06

---
### P26 — gpt-oss-120b
## Resposta
No recorte analisado, os clientes destacam que a configuração de dispositivos Wi‑Fi pode ser **complicada sem suporte técnico** [R1] e que **orientações ou vídeos de instalação facilitam o processo** [R5]. Também há relatos de que alguns produtos são **fáceis de instalar** [R13, R15] e que a experiência pode ser **rápida e prática** [R11].

## O que os clientes relatam
- **Dificuldade sem apoio técnico:** “configurar o WiFi Repeater é complicado só com auxílio de um técnico” [R1]  
- **Ajuda via orientação/vídeo:** “vendedor entrar em contato e dar orientação e filme de instalação do produto” [R5]  
- **Instalação simples:** “Facil de instalar” [R13]  
- **Instalação fácil e com proteção:** “fácil instalação e com proteção de qualidade (IP67)” [R15]  
- **Processo rápido e prático:** “Rapido, simples e pratico” [R11]  

## Interpretação
Isso sugere que, para melhorar a experiência de configuração, a empresa deve disponibilizar materiais de apoio (vídeos, manuais) e, se possível, oferecer suporte técnico. Produtos que já são percebidos como fáceis de instalar reforçam a importância de um design intuitivo.

## Limitações
A resposta se baseia em 5 avaliações recuperadas de um recorte de 42 251 avaliações com texto e não mede frequências na base.

In [7]:
# ===== Célula 6 — Amostra de afirmações para validar o juiz de fidelidade (sem o veredito dele) =====
import random
import textwrap

from src.rag.geracao import formatar_contexto

arquivo_fid = RAIZ / "resultados" / "fidelidade" / "densa_k20_prompt_v2.jsonl"
fidelidade = [json.loads(l) for l in arquivo_fid.read_text(encoding="utf-8").splitlines()
              if l.strip()]
fidelidade = [f for f in fidelidade if "erro" not in f]
resp_por_chave = {(r["id"], r["modelo"]): r for r in registros if "erro" not in r}

afirmacoes = []
for f in fidelidade:
    sinalizada = len(resp_por_chave[(f["id"], f["modelo"])]["trechos_nao_encontrados"]) > 0
    for a in f["afirmacoes"]:
        afirmacoes.append({**a, "id": f["id"], "modelo": f["modelo"],
                           "resposta_sinalizada": sinalizada})

random.seed(7)  # sorteio reprodutível
sust_sinalizadas = [a for a in afirmacoes if a["veredito"] == "sustentada" and a["resposta_sinalizada"]]
sust_limpas = [a for a in afirmacoes if a["veredito"] == "sustentada" and not a["resposta_sinalizada"]]
outras = [a for a in afirmacoes if a["veredito"] != "sustentada"]
amostra_fid = (random.sample(sust_sinalizadas, min(6, len(sust_sinalizadas)))
               + random.sample(sust_limpas, min(3, len(sust_limpas)))
               + random.sample(outras, min(3, len(outras))))
random.shuffle(amostra_fid)

for i, a in enumerate(amostra_fid, start=1):
    r = resp_por_chave[(a["id"], a["modelo"])]
    evid = pd.DataFrame(r["evidencias"])
    citadas = evid[evid["ref"].isin(a["citacoes"])]
    print(f"\n#{i:02d} [{a['id']}] PERGUNTA: {r['pergunta']}")
    print(f"     AFIRMAÇÃO: {a['afirmacao']}  {a['citacoes']}")
    print("     AVALIAÇÕES CITADAS:")
    corpo = formatar_contexto(citadas) if not citadas.empty else "(nenhuma das citações existe no contexto)"
    print(textwrap.indent(corpo, "       "))


#01 [P02] PERGUNTA: O que os clientes relatam sobre problemas relacionados à entrega?
     AFIRMAÇÃO: problema de comunicação entre os e‑mail enviados  ['R8']
     AVALIAÇÕES CITADAS:
       [R8] nota 4 | categoria: informatica acessorios | entrega: no prazo | compra: 2018-06 | UF: SP
       "Recomendo. Houve um problema de comunicação entre os e-mail enviados e s/ a entrega em separado sem aviso. mas no final recebi tudo certo e no prazo. Assim sendo, recomendo e compraria novamente"

#02 [P09] PERGUNTA: O que os clientes relatam sobre produtos que chegaram quebrados ou danificados?
     AFIRMAÇÃO: Alguns clientes conseguem solução rápida após reclamação, enquanto outros não obtêm reposição.  ['R2', 'R3', 'R4', 'R16']
     AVALIAÇÕES CITADAS:
       [R2] nota 5 | categoria: papelaria | entrega: no prazo | compra: 2017-03 | UF: MG
       "O Primeiro produto veio com defeito. Depois que reclamei enviaram outro rapidamente, em perfeito estado e ainda levei um brinde. Pode confiar, eles 

In [8]:
# ===== Célula 7 — Seus vereditos e a comparação com o juiz de fidelidade =====
# S = sustentada, P = parcial, N = não sustentada (na ordem #01 a #12)
vereditos_humanos = ["S", "S", "S", "S", "S", "N",
                     "S", "S", "P", "P", "S", "S"]

mapa = {"S": "sustentada", "P": "parcial", "N": "nao_sustentada"}
assert len(vereditos_humanos) == len(amostra_fid), "um veredito para cada afirmação"
assert all(v in mapa for v in vereditos_humanos), "use apenas S, P ou N"

humano = [mapa[v] for v in vereditos_humanos]
juiz_v = [a["veredito"] for a in amostra_fid]
comparacao = pd.DataFrame({"n": range(1, len(amostra_fid) + 1),
                           "id": [a["id"] for a in amostra_fid],
                           "modelo": [a["modelo"].split("/")[-1] for a in amostra_fid],
                           "afirmacao": [a["afirmacao"] for a in amostra_fid],
                           "humano": humano, "juiz": juiz_v})

exata = (comparacao["humano"] == comparacao["juiz"]).mean()
binaria = ((comparacao["humano"] == "sustentada") == (comparacao["juiz"] == "sustentada")).mean()
juiz_sust = comparacao[comparacao["juiz"] == "sustentada"]
leniencia = (juiz_sust["humano"] != "sustentada").mean() if len(juiz_sust) else float("nan")

print(f"Concordância exata:                         {exata:.0%}")
print(f"Concordância 'sustentada ou não':           {binaria:.0%}")
print(f"Leniência (juiz disse sustentada, você não): {leniencia:.0%} de {len(juiz_sust)} casos")
print("\nMatriz (linhas = você, colunas = juiz):")
print(pd.crosstab(comparacao["humano"], comparacao["juiz"], rownames=["você"], colnames=["juiz"]))
print("\nDiscordâncias:")
for _, d in comparacao[comparacao["humano"] != comparacao["juiz"]].iterrows():
    print(f"  #{d['n']:02d} [{d['id']}] {d['modelo']:13s} você {d['humano']} x juiz {d['juiz']} | {d['afirmacao'][:80]}")

comparacao.to_csv(RAIZ / "resultados" / "validacao_fidelidade.csv", index=False, encoding="utf-8")
print("\nValidação salva em resultados/validacao_fidelidade.csv")

Concordância exata:                         67%
Concordância 'sustentada ou não':           67%
Leniência (juiz disse sustentada, você não): 22% de 9 casos

Matriz (linhas = você, colunas = juiz):
juiz            nao_sustentada  parcial  sustentada
você                                               
nao_sustentada               1        0           0
parcial                      0        0           2
sustentada                   1        1           7

Discordâncias:
  #04 [P15] gpt-oss-120b  você sustentada x juiz nao_sustentada | Sugestão para sair da logística dos correios pois o serviço é um lixo.
  #07 [P02] gpt-oss-120b  você sustentada x juiz parcial | Atrasado (15 dias após o prazo)
  #09 [P16] gpt-oss-120b  você parcial x juiz sustentada | Produto de qualidade, entrega antes do prazo previsto. Gostei.
  #10 [P03] qwen3.8-27b   você parcial x juiz sustentada | O processo de compra é descrito como 'fácil' e 'eficiente'.

Validação salva em resultados/validacao_fidelidade.csv


In [10]:
# ===== Célula 8 — Consenso da validação de fidelidade =====
from src.avaliacao.relatorio import contem_trecho_sinalizado

consenso = list(vereditos_humanos)
ajustes_fid = {2: "P", 7: "P", 9: "S", 10: "S", 12: "P"}  # número da afirmação: veredito de consenso
for n, v in ajustes_fid.items():
    consenso[n - 1] = v

comparacao["consenso"] = [mapa[v] for v in consenso]
comparacao["pego_pelo_verificador"] = [
    contem_trecho_sinalizado(a["afirmacao"],
                             resp_por_chave[(a["id"], a["modelo"])]["trechos_nao_encontrados"])
    for a in amostra_fid]

exata_c = (comparacao["consenso"] == comparacao["juiz"]).mean()
sust_juiz = comparacao[comparacao["juiz"] == "sustentada"]
lenientes = sust_juiz[sust_juiz["consenso"] != "sustentada"]

print(f"Juiz x consenso (exata):           {exata_c:.0%}")
print(f"Leniência do juiz:                 {len(lenientes)} de {len(sust_juiz)} "
      f"({len(lenientes) / len(sust_juiz):.0%})")
print(f"  ...pegos pelo verificador:       {int(lenientes['pego_pelo_verificador'].sum())}")
print(f"Juiz exigente demais:              "
      f"{((comparacao['juiz'] != 'sustentada') & (comparacao['consenso'] == 'sustentada')).sum()}")
print("\nCasos lenientes:")
for _, d in lenientes.iterrows():
    print(f"  #{d['n']:02d} [{d['id']}] {d['modelo']:13s} pego pelo verificador: "
          f"{'sim' if d['pego_pelo_verificador'] else 'não'} | {d['afirmacao'][:80]}")

comparacao.to_csv(RAIZ / "resultados" / "validacao_fidelidade.csv", index=False, encoding="utf-8")
print("\nValidação (com consenso) salva em resultados/validacao_fidelidade.csv")

Juiz x consenso (exata):           75%
Leniência do juiz:                 2 de 9 (22%)
  ...pegos pelo verificador:       1
Juiz exigente demais:              1

Casos lenientes:
  #02 [P09] gpt-oss-120b  pego pelo verificador: não | Alguns clientes conseguem solução rápida após reclamação, enquanto outros não ob
  #12 [P17] gpt-oss-120b  pego pelo verificador: sim | não me dão soluções… não consigo fazer a troca

Validação (com consenso) salva em resultados/validacao_fidelidade.csv
